In [1]:
import random
import itertools
import math
from typing import List, NamedTuple, Tuple, Union
 
import numpy as np
import pandas as pd

Starting assumptions :
- a card is defined by 3 parameters : its level, its trigger number and whether it's climax or not
- I didn't took into factor colors and events yet

In [2]:
class Card(NamedTuple):
    level: int
    n_triggers: int
    is_climax: bool

For deck selection, we procede in 3 steps :
- first, we define the quantities of cards for each [level, n_triggers, is_climax] combination
- we parse these lists to create pools and check for incoherences
- we create a random deck with these card pools according to selected parameters
  Note : the system doesn't use any damage process which looks at specific cards in both decks yet, but I decided to implement it earlier for future stages

In [3]:
class Pools(NamedTuple):
    main_climax: List[Card]
    main_non_climax: List[Card]
    trigger_climax: List[Card]
    trigger_non_climax: List[Card]

In [4]:
def expand_pool(card_counts: dict) -> List[Card]:
    return [card for card, count in card_counts.items() for _ in range(count)]

In [5]:
def build_pools(main_non_climax_specs, main_climax_specs,
                 trigger_non_climax_specs, trigger_climax_specs) -> Pools:
    return Pools(
        main_climax=expand_pool(main_climax_specs),
        main_non_climax=expand_pool(main_non_climax_specs),
        trigger_climax=expand_pool(trigger_climax_specs),
        trigger_non_climax=expand_pool(trigger_non_climax_specs),
    )

- Simulation initial parameters.
- The system doesn't use the card level yet.
- The goal was to implement it early for future usages.

In [6]:
MAIN_NON_CLIMAX_SPECS = {
    Card(0, 0, False): 8,
    Card(0, 1, False): 6,
    Card(1, 0, False): 8,
    Card(1, 1, False): 6,
    Card(2, 0, False): 8,
    Card(2, 1, False): 6,
    Card(3, 0, False): 8,
    Card(3, 1, False): 4,
}
MAIN_CLIMAX_SPECS = {
    Card(0, 0, True): 4,
    Card(0, 1, True): 4,
    Card(0, 2, True): 4,
}

In [29]:
TRIGGER_NON_CLIMAX_SPECS = {
    Card(0, 0, False): 17,
    Card(0, 1, False): 0,
    Card(1, 0, False): 12,
    Card(1, 1, False): 0,
    Card(2, 0, False): 0,
    Card(2, 1, False): 3,
    Card(3, 0, False): 3,
    Card(3, 2, False): 7,
}
TRIGGER_CLIMAX_SPECS = {
    Card(0,0, True): 4,
	Card(0,1, True): 4,
	Card(0,2, True): 0,
}

In [30]:
TRIGGER_DECK_SIZE = 16
TRIGGER_DECK_CLIMAX = 0
 
MAIN_DECK_SIZES = [40, 50]
CLIMAX_NUMBERS = [6, 8]

In [31]:
OCCURRENCES: List[Union[int, str]] = [1, "2A", 1, "3A", 1]
 
N_SIMULATIONS = 20000
OUT_PATH = "resultats_simulation_v2.csv"
 
random.seed(42)

- This function checks here if your deck ratios are coherent, regarding your demands.

In [32]:
def validate_pools(pools: Pools, deck_size_list, climax_list,
                     trigger_deck_size, trigger_deck_climax) -> None:
    needed = {
        "main_climax": max(climax_list),
        "main_non_climax": max(s - c for s in deck_size_list for c in climax_list if c < s),
        "trigger_climax": trigger_deck_climax,
        "trigger_non_climax": trigger_deck_size - trigger_deck_climax,
    }
    for field, n_needed in needed.items():
        pool = getattr(pools, field)
        if len(pool) < n_needed:
            raise ValueError(
                f"Le pool '{field}' ne contient que {len(pool)} exemplaires, "
                f"or il en faut au moins {n_needed}. Augmente le nombre "
                f"d'exemplaires par carte dans les specs correspondantes."
            )

In [33]:
validate_pools(
    build_pools(MAIN_NON_CLIMAX_SPECS, MAIN_CLIMAX_SPECS, TRIGGER_NON_CLIMAX_SPECS, TRIGGER_CLIMAX_SPECS),
    MAIN_DECK_SIZES, CLIMAX_NUMBERS, TRIGGER_DECK_SIZE, TRIGGER_DECK_CLIMAX,
)

Before each simulation, the system builds different decks from the card pools to use.

In [34]:
def build_deck(climax_pool, non_climax_pool, size, climax_n) -> List[Card]:
    deck = random.sample(climax_pool, climax_n) + random.sample(non_climax_pool, size - climax_n)
    random.shuffle(deck)
    return deck

Once we build the decks, we parse them to separate effect damages from attack damages.

In [35]:
def parse_occurrence(spec: Union[int, str]) -> Tuple[int, bool]:
    s = str(spec).strip().upper()
    return (int(s[:-1]), True) if s.endswith("A") else (int(s), False)

def parse_occurrences(occurrences) -> List[Tuple[int, bool]]:
    return [parse_occurrence(spec) for spec in occurrences]

All game rules are implemented there.

In [36]:
class GameSystem:
     
    def __init__(self, main_deck: List[Card], trigger_deck: List[Card]):
        self.main_deck = main_deck
        self.discard: List[Card] = []
        self.trigger_deck = trigger_deck
        self.trigger_discard: List[Card] = []
        self.clock_zone: List[Card] = []
        self.level_zone: List[Card] = [] 
        self.total_damage = 0
        self.initial_main_count = len(main_deck)
 
    # Trigger check rule
    def trigger_check(self) -> int:
        if not self.trigger_deck:
            if not self.trigger_discard:
                return 0
            self.trigger_deck.extend(self.trigger_discard)
            self.trigger_discard.clear()
            random.shuffle(self.trigger_deck)
        card = self.trigger_deck.pop()
        self.trigger_discard.append(card)
        return card.n_triggers if card.n_triggers <= 2 else 0
 
    # Level up rule
    def levelup(self) -> None:
        while len(self.clock_zone) >= 7:
            batch = self.clock_zone[:7]
            del self.clock_zone[:7]
            idx = next((i for i, c in enumerate(batch) if not c.is_climax), None)
            if idx is not None:
                self.level_zone.append(batch.pop(idx))
            self.discard.extend(batch)
 
    # Refresh penalty rule
    def refresh(self) -> bool:
        if not self.discard:
            return True
        new_deck = self.discard[:]
        self.discard.clear()
        random.shuffle(new_deck)
        self.clock_zone.append(new_deck.pop())
        self.total_damage += 1
        self.levelup()
        self.main_deck.extend(new_deck)
        return False

    def keep_cards(self, cards: List[Card]) -> None:
        self.clock_zone.extend(cards)
        self.total_damage += len(cards)
        self.levelup()
 
    def resolve_occurrence(self, draw_count: int, is_trigger: bool) -> bool:
        if is_trigger:
            draw_count += self.trigger_check()
 
        drawn: List[Card] = []
        while len(drawn) < draw_count:
            if not self.main_deck:
                if self.refresh():
                    return True
                continue
            card = self.main_deck.pop()
            drawn.append(card)
            if card.is_climax:
                self.discard.extend(drawn)
                return False
 
        self.keep_cards(drawn)
        return False
 
    def run(self, parsed_occurrences: List[Tuple[int, bool]]) -> int:
        for draw_count, is_trigger in parsed_occurrences:
            if self.resolve_occurrence(draw_count, is_trigger):
                break
        return self.total_damage
 
    # This variable is used here to verify no card is lost during the simulations.
    # It will be desactived by definition, but it is relevant to check for bugs.
    def check_invariant(self) -> None:
        tracked = (len(self.main_deck) + len(self.discard)
                   + len(self.clock_zone) + len(self.level_zone))
        if tracked != self.initial_main_count:
            raise AssertionError(
                f"Incoherence de conservation des cartes : {tracked} cartes "
                f"trackees (main_deck+discard+clock_zone+level_zone) "
                f"pour {self.initial_main_count} au depart."
            )

In [37]:
def run_single_trial(deck_size, climax_n, parsed_occurrences, pools: Pools,
                      trigger_deck_size, trigger_deck_climax,
                      check_invariants=False) -> int:
    main_deck = build_deck(pools.main_climax, pools.main_non_climax, deck_size, climax_n)
    trigger_deck = build_deck(pools.trigger_climax, pools.trigger_non_climax,
                               trigger_deck_size, trigger_deck_climax)
    state = GameSystem(main_deck, trigger_deck)
    damage = state.run(parsed_occurrences)
    if check_invariants:
        state.check_invariant()
    return damage

In [38]:
def max_possible_damage(occurrences, deck_size_list=None) -> int:
    base = sum(draw_count + (2 if is_trigger else 0) for draw_count, is_trigger in parse_occurrences(occurrences))
    if not deck_size_list:
        return base
    max_refreshes = math.ceil(base / min(deck_size_list))
    return base + max_refreshes

In [39]:
def generate_table(deck_size_list, climax_list, occurrences, n_trials,
                    main_non_climax_specs=MAIN_NON_CLIMAX_SPECS,
                    main_climax_specs=MAIN_CLIMAX_SPECS,
                    trigger_non_climax_specs=TRIGGER_NON_CLIMAX_SPECS,
                    trigger_climax_specs=TRIGGER_CLIMAX_SPECS,
                    trigger_deck_size=TRIGGER_DECK_SIZE,
                    trigger_deck_climax=TRIGGER_DECK_CLIMAX,
                    max_damage_column=None,
                    out_path=OUT_PATH,
                    seed=None,
                    check_invariants=False) -> pd.DataFrame:
    
    """
    Parameters
    ----------
    deck_size_list     : deck sizes to test
    climax_list        : climax counts to test [6, 8]
    occurrences        : damage sequence to test (type '1A' with to get soul trigger check)
    n_trials           : Monte Carlo simulations number for each (deck_size_list, climax_list) combination
    main_non_climax_specs, main_climax_specs,
    trigger_non_climax_specs, trigger_climax_specs
                        : customize both decks as you wish
    trigger_deck_size  : trigger deck size
    trigger_deck_climax: climax count in trigger deck
    max_damage_column  : table construction parameter, used to account to account for triggers and refresh damages
    out_path           : csv.table path on your PC
    seed               : initialisation number
    check_invariants   : debugger system here, desactived by default
    """
    if seed is not None:
        random.seed(seed)
 
    pools = build_pools(main_non_climax_specs, main_climax_specs,
                         trigger_non_climax_specs, trigger_climax_specs)
    validate_pools(pools, deck_size_list, climax_list, trigger_deck_size, trigger_deck_climax)
    parsed_occurrences = parse_occurrences(occurrences)
 
    thresholds = np.arange((max_damage_column or max_possible_damage(occurrences, deck_size_list)) + 1)
    columns = [f"P(total_damage>={t})" for t in thresholds]
    rows = []
 
    for size, climax_n in itertools.product(deck_size_list, climax_list):
        if climax_n >= size:
            continue
        results = np.fromiter(
            (run_single_trial(size, climax_n, parsed_occurrences, pools,
                               trigger_deck_size, trigger_deck_climax, check_invariants)
             for _ in range(n_trials)),
            dtype=int, count=n_trials,
        )
        probs = (results[:, None] >= thresholds[None, :]).mean(axis=0).round(4)
        rows.append([size, climax_n, *probs])
 
    table = pd.DataFrame(rows, columns=["main_deck_size", "climax_number"] + columns)
 
    if out_path:
        table.to_csv(out_path, index=False)
        print(f"CSV ecrit : {out_path} ({len(rows)} combinaisons x {n_trials} simulations)")
 
    return table

In [40]:
table = generate_table(
    deck_size_list=[20, 25, 30],
    climax_list=[4, 6, 8],
    occurrences=[4, '3A', 4, '3A', 4, '3A'],
    n_trials=20000,
    main_non_climax_specs = {
    Card(0, 0, False): 17,
    Card(0, 1, False): 0,
    Card(1, 0, False): 12,
    Card(1, 1, False): 0,
    Card(2, 0, False): 0,
    Card(2, 1, False): 3,
    Card(3, 0, False): 3,
    Card(3, 1, False): 7,
},

    main_climax_specs = {
    Card(0,0, True): 4,
	Card(0,1, True): 4,
	Card(0,2, True): 0,
},

    trigger_non_climax_specs = {
    Card(0, 0, False): 17,
    Card(0, 1, False): 0,
    Card(1, 0, False): 12,
    Card(1, 1, False): 0,
    Card(2, 0, False): 0,
    Card(2, 1, False): 3,
    Card(3, 0, False): 3,
    Card(3, 1, False): 7,
},

    trigger_climax_specs = {
    Card(0,0, True): 4,
	Card(0,1, True): 4,
	Card(0,2, True): 0,
},)
table.to_csv('resultats_simulation_v2.csv', index=False)

table.head()

CSV ecrit : resultats_simulation_v2.csv (9 combinaisons x 20000 simulations)


,main_deck_size,climax_number,P(total_damage>=0),P(total_damage>=1),P(total_damage>=2),P(total_damage>=3),P(total_damage>=4),P(total_damage>=5),P(total_damage>=6),P(total_damage>=7),...,P(total_damage>=20),P(total_damage>=21),P(total_damage>=22),P(total_damage>=23),P(total_damage>=24),P(total_damage>=25),P(total_damage>=26),P(total_damage>=27),P(total_damage>=28),P(total_damage>=29)
0,20,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,0.9982,0.9235,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,20,6,1.0,0.9427,0.9427,0.9427,0.7410,0.4928,0.4928,0.4043,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,20,8,1.0,0.6796,0.6796,0.6796,0.3904,0.1314,0.1314,0.0970,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,25,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9716,...,0.0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,25,6,1.0,0.9873,0.9873,0.9873,0.8985,0.7856,0.7856,0.7032,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
